In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM  # Punkt 4 benötigt
import pickle


# ============================================
# 1. Funktion zum Parsen der Antwort in yes/no/maybe
# ============================================
def parse_answer(text: str) -> str:
    text = text.strip().lower()
    if "yes" in text:
        return "yes"
    if "no" in text:
        return "no"
    if "maybe" in text:
        return "maybe"
    return "unknown"

# ============================================
# 2. Fragen laden
# ============================================
print("Lade Fragen mit Ground Truth...")
df_questions = pd.read_pickle('../../Dataset/golden69_questions.pkl')   # Pfad anpassen

# Sicherstellen, dass die Spalte 'final_decision' existiert und yes/no/maybe enthält
if 'final_decision' not in df_questions.columns:
    raise ValueError("Die Datei enthält keine Spalte 'final_decision'.")

# ============================================
# 3. Evaluation
# ============================================
true_labels = []
pred_labels = []

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    question = row['question']
    true = row['final_decision'].strip().lower()
    print(question)

    # Retrieval-Chunks und Metadaten abrufen
    gefundene_chunks, metadata = retrieve_with_metadata(question, k=5)
    # Optional: ersten gefundenen Titel ausgeben
    if metadata:
        first_title = metadata[0]['title']
        print(f'Metadata: {first_title}')
    else:
        first_title = ""

    try:
        answer_text = generate_answer(question)   # hier wird der neue Prompt verwendet
        pred = parse_answer(answer_text)
    except Exception as e:
        pred = "unknown"
        answer_text = str(e)

    # Debugging bei unknown oder falscher Antwort
    if pred == "unknown" or (pred != true):
        print(f"\nFrage: {question[:80]}...")
        print(f"  Erster retrieved Titel: {first_title}")
        print(f"  Generierte Antwort: {answer_text}")
        print(f"  True: {true} | Pred: {pred}")

    true_labels.append(true)
    pred_labels.append(pred)

# ============================================
# 4. Metriken berechnen und ausgeben
# ============================================
# Nur gültige yes/no/maybe Vorhersagen
valid_indices = [i for i, p in enumerate(pred_labels) if p in ['yes', 'no', 'maybe']]
true_valid = [true_labels[i] for i in valid_indices]
pred_valid = [pred_labels[i] for i in valid_indices]

# Metriken für alle drei Klassen
labels = ['yes', 'no', 'maybe']
accuracy = accuracy_score(true_valid, pred_valid)
precision, recall, f1, _ = precision_recall_fscore_support(true_valid, pred_valid, labels=labels, average=None)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(true_valid, pred_valid, average='macro')
cm = confusion_matrix(true_valid, pred_valid, labels=labels)

print("\n" + "="*60)
print("EVALUATIONSERGEBNISSE")
print("="*60)
print(f"Anzahl ausgewerteter Fragen: {len(df_questions)}")
print(f"Davon 'unknown'-Antworten (nicht parsebar): {pred_labels.count('unknown')}")
print(f"Gültige Vergleiche (yes/no/maybe): {len(true_valid)}")
print(f"\nAccuracy (über alle gültigen): {accuracy:.4f}")
print("\nKlassenweise Metriken:")
for i, label in enumerate(labels):
    print(f"  {label.upper()}  - Precision: {precision[i]:.4f}, Recall: {recall[i]:.4f}, F1: {f1[i]:.4f}")
print(f"\nMakro-Durchschnitt (ungewichtet): Precision: {macro_precision:.4f}, Recall: {macro_recall:.4f}, F1: {macro_f1:.4f}")
print("\nConfusion Matrix (Zeilen = wahr, Spalten = vorhergesagt):")
print("         yes   no   maybe")
print("       +-----------------")
for i, label in enumerate(labels):
    row = cm[i]
    print(f" {label:5} | {row[0]:3}  {row[1]:3}  {row[2]:3}")

# ============================================
# 5. Optional: Ergebnisse speichern
# ============================================
df_results = pd.DataFrame({
    'question': df_questions['question'],
    'true_label': true_labels,
    'pred_label': pred_labels
})
df_results.to_csv('rag_evaluation_results.csv', index=False)
print("\nErgebnisse wurden in 'rag_evaluation_results.csv' gespeichert.")






